In [1]:
# !pip install yfinance crewai crewai-tools

In [2]:
import re
import json
import os
import yfinance as yf
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import FileReadTool

In [3]:
from pydantic import BaseModel, Field

class QueryAnalysisOutput(BaseModel):
    """Structured output for the query analysis task."""
    symbol: str = Field(..., description="Stock ticker symbol (e.g., TSLA, AAPL).")
    timeframe: str = Field(..., description="Time period (e.g., '1d', '1mo', '1y').")
    action: str = Field(..., description="Action to be performed (e.g., 'fetch', 'plot').")

In [8]:
# Required environment variables:
# AZURE_OPENAI_API_KEY
# AZURE_OPENAI_ENDPOINT (example: https://<resource-name>.openai.azure.com/openai/v1)
# AZURE_OPENAI_DEPLOYMENT_NAME (your model deployment name in Azure OpenAI)

api_key = os.getenv('AZURE_OPENAI_API_KEY', '').strip()
endpoint = os.getenv('AZURE_OPENAI_ENDPOINT', '').strip().rstrip('/')
deployment = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME', '').strip()

missing = [
    name for name, value in {
        'AZURE_OPENAI_API_KEY': api_key,
        'AZURE_OPENAI_ENDPOINT': endpoint,
        'AZURE_OPENAI_DEPLOYMENT_NAME': deployment,
    }.items() if not value
]
if missing:
    raise ValueError(f"Missing required environment variables: {', '.join(missing)}")

# Use Azure OpenAI OpenAI-compatible endpoint to avoid CrewAI native Azure provider dependency
llm = LLM(
    model=deployment,
    api_key=api_key,
    base_url=endpoint,
    # temperature=0.7
)

# 1) Query parser agent
query_parser_agent = Agent(
    role='Stock Data Analyst',
    goal='Extract stock details and fetch required data from this user query: {query}.',
    backstory='You are a financial analyst specializing in stock market data retrieval.',
    llm=llm,
    verbose=True,
    memory=True,
)

query_parsing_task = Task(
    description='Analyze the user query and extract stock details.',
    expected_output="A dictionary with keys: 'symbol', 'timeframe', 'action'.",
    output_pydantic=QueryAnalysisOutput,
    agent=query_parser_agent,
)

# 2) Code writer agent
code_writer_agent = Agent(
    role='Senior Python Developer',
    goal='Write Python code to visualize stock data.',
    backstory=(
        'You are a Senior Python developer specializing in stock market data visualization. '
        'You are also a Pandas, Matplotlib and yfinance library expert. '
        'You are skilled at writing production-ready Python code'
    ),
    llm=llm,
    verbose=True,
)

code_writer_task = Task(
    description=(
        'Write Python code to visualize stock data based on the inputs from the stock analyst '
        'where you would find stock symbol, timeframe and action.'
    ),
    expected_output='A clean and executable Python script file (.py) for stock visualization.',
    agent=code_writer_agent,
)

# 3) Code execution agent
code_execution_agent = Agent(
    role='Senior Code Execution Expert',
    goal='Review and execute the generated Python code by code writer agent to visualize stock data.',
    backstory='You are a code execution expert. You are skilled at executing Python code.',
    allow_code_execution=True,
    llm=llm,
    verbose=True,
)

code_execution_task = Task(
    description='Review and execute the generated Python code by code writer agent to visualize stock data.',
    expected_output='A clean and executable Python script file (.py) for stock visualization.',
    agent=code_execution_agent,
)

In [9]:
### --- CREW SETUP --- ###

crew = Crew(
    agents=[query_parser_agent, code_writer_agent, code_execution_agent],
    tasks=[query_parsing_task, code_writer_task, code_execution_task],
    process=Process.sequential
)

# In notebooks, use async kickoff to avoid event-loop errors
result = await crew.kickoff_async(inputs={'query': 'Plot YTD stock gain of Tesla'})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Task: Analyze the user query and extract stock details.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "symbol": "TSLA",                                                                                            │
│    "timeframe": "YTD",                                                                                          │
│    "action": "plot"                                                                                             │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'memory_save_failed' closed 'agent_execution_started' (expected 
'memory_save_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'task_started' (expected 
'agent_execution_started')

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Developer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  import yfinance as yf                                                                                          │
│  import pandas as pd                                                                                            │
│  import matplotlib.pyplot as plt                                                                                │
│  from datetime import datetime                                                                                  │
│                                                                                                                 │
│  # ==== USER INPUT (from analyst) ====                                                                          │
│  symbol = "TSLA"                                                                                                │
│  timeframe = "YTD"                                                                                              │
│  action = "plot"                                                                                                │
│                                                                                                                 │
│  # ==== HELPER FUNCTION TO GET PERIOD DATES ====                                                                │
│                                                                                                                 │
│  def get_period_dates(timeframe):                                                                               │
│      today = datetime.today()                                                                                   │
│      if timeframe.upper() == "YTD":                                                                             │
│          start_date = datetime(today.year, 1, 1)                                                                │
│          end_date = today                                                                                       │
│          return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")                                  │
│      else:                                                                                                      │
│          raise ValueError("Currently only 'YTD' timeframe is supported.")                                       │
│                                                                                                                 │
│  # ==== MAIN LOGIC ====                                                                                         │
│                                                                                                                 │
│  def fetch_stock_data(symbol, start_date, end_date):                                                            │
│      ticker = yf.Ticker(symbol)                                                                                 │
│      df = ticker.history(start=start_date, end=end_date)                                                        │
│      if df.empty:                                                                                               │
│          raise ValueError(f"No data found for {symbol} from {start_date} to {end_date}.")                       │
│      return df                                         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Task: Review and execute the generated Python code by code writer agent to visualize stock data.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  import yfinance as yf                                                                                          │
│  import pandas as pd                                                                                            │
│  import matplotlib.pyplot as plt                                                                                │
│  from datetime import datetime                                                                                  │
│                                                                                                                 │
│  # ==== USER INPUT (from analyst) ====                                                                          │
│  symbol = "TSLA"                                                                                                │
│  timeframe = "YTD"                                                                                              │
│  action = "plot"                                                                                                │
│                                                                                                                 │
│  # ==== HELPER FUNCTION TO GET PERIOD DATES ====                                                                │
│                                                                                                                 │
│  def get_period_dates(timeframe):                                                                               │
│      today = datetime.today()                                                                                   │
│      if timeframe.upper() == "YTD":                                                                             │
│          start_date = datetime(today.year, 1, 1)                                                                │
│          end_date = today                                                                                       │
│          return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")                                  │
│      else:                                                                                                      │
│          raise ValueError("Currently only 'YTD' timeframe is supported.")                                       │
│                                                                                                                 │
│  # ==== MAIN LOGIC ====                                                                                         │
│                                                                                                                 │
│  def fetch_stock_data(symbol, start_date, end_date):                                                            │
│      ticker = yf.Ticker(symbol)                                                                                 │
│      df = ticker.history(start=start_date, end=end_date)                                                        │
│      if df.empty:                                                                                               │
│          raise ValueError(f"No data found for {symbol} from {start_date} to {end_date}.")                       │
│      return df                                         

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_completed' emitted with empty scope stack. Missing starting 
event?

In [10]:
print(result.raw)

```python
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# ==== USER INPUT (from analyst) ====
symbol = "TSLA"
timeframe = "YTD"
action = "plot"

# ==== HELPER FUNCTION TO GET PERIOD DATES ====

def get_period_dates(timeframe):
    today = datetime.today()
    if timeframe.upper() == "YTD":
        start_date = datetime(today.year, 1, 1)
        end_date = today
        return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")
    else:
        raise ValueError("Currently only 'YTD' timeframe is supported.")

# ==== MAIN LOGIC ====

def fetch_stock_data(symbol, start_date, end_date):
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=start_date, end=end_date)
    if df.empty:
        raise ValueError(f"No data found for {symbol} from {start_date} to {end_date}.")
    return df

def plot_stock_data(df, symbol, timeframe):
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df['Close'], label='Close Price'